# 05-02 Advanced Pair Analysis

Interactive dashboard for the correlation-pair strategy: ranking (Plotly), trade-by-trade drilldown, threshold-sensitivity heatmap, rolling-IC stability, and CSV/HTML report export. Data is loaded from S3 via DuckDB httpfs.

In [ ]:
# ============================================================================
# SETUP -- installs, imports, config (env vars / config.json -- never hardcoded)
# ============================================================================

# --- Install packages (no-op if already present) --------------------------
# !pip install -q duckdb scipy plotly ipywidgets matplotlib --upgrade
import json
import os
from io import BytesIO
from pathlib import Path

import boto3
import duckdb
import ipywidgets as widgets
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from IPython.display import clear_output, display
from plotly.subplots import make_subplots

# --- Configuration --------------------------------------------------------
# Secrets resolve in priority order:
#   1. Environment variables (AWS_ACCESS_KEY_ID, AWS_SECRET_ACCESS_KEY,
#      AWS_REGION, S3_BUCKET, MASSIVE_API_KEY, ...)
#   2. config.json in the current directory (see config.example.json)
#   3. Built-in defaults (non-secret values only)
# On Kaggle: set secrets via notebook settings (Add-ons -> Secrets), which
# are injected as environment variables.
CONFIG_FILE = "config.json"


def get_secret(name, default=""):
    val = os.environ.get(name)
    if val:
        return val
    if Path(CONFIG_FILE).exists():
        try:
            with open(CONFIG_FILE) as f:
                data = json.load(f)
            if name in data:
                return str(data[name])
        except (OSError, ValueError):
            pass
    return default


class Config:
    def __init__(self):
        self.aws_access_key_id = ""
        self.aws_secret_access_key = ""
        self.aws_region = "us-east-1"
        self.s3_bucket = "market-data-zw"
        self.massive_api_key = ""

        # Paths (S3 keys under the bucket)
        self.types_prefix = "parquet_data/types"
        self.tickers_prefix = "parquet_data/summary/tickers"
        self.ticker_details_prefix = "parquet_data/summary/ticker_yahoo_details"
        self.minute_staging_prefix = "parquet_data/minute_data_staging"
        self.minute_final_prefix = "parquet_data/minute_data_final"
        self.minute_summary_prefix = "parquet_data/summary/minute_summary"
        self.daily_volume_prefix = "parquet_data/summary/daily_volume"
        self.correlation_prefix = "parquet_data/strategies/correlation"
        self.backtest_prefix = "parquet_data/backtest"
        self.backtest_metrics_prefix = "parquet_data/analysis/backtest_metrics"

        # Spark
        self.spark_executor_memory = "24g"
        self.spark_executor_cores = 4
        self.spark_driver_memory = "24g"
        self.spark_tmp = "/tmp/spark"


def load_config():
    cfg = Config()
    if Path(CONFIG_FILE).exists():
        try:
            with open(CONFIG_FILE) as f:
                data = json.load(f)
            for key, value in data.items():
                if hasattr(cfg, key):
                    setattr(cfg, key, value)
        except (OSError, ValueError) as e:
            print(f"[config] WARNING: could not load {CONFIG_FILE}: {e}")

    env_map = {
        "AWS_ACCESS_KEY_ID": "aws_access_key_id",
        "AWS_SECRET_ACCESS_KEY": "aws_secret_access_key",
        "AWS_REGION": "aws_region",
        "S3_BUCKET": "s3_bucket",
        "MASSIVE_API_KEY": "massive_api_key",
        "SPARK_DRIVER_MEMORY": "spark_driver_memory",
        "SPARK_EXECUTOR_MEMORY": "spark_executor_memory",
        "SPARK_EXECUTOR_CORES": "spark_executor_cores",
    }
    for env_name, attr in env_map.items():
        val = os.environ.get(env_name)
        if val:
            if attr == "spark_executor_cores":
                val = int(val)
            setattr(cfg, attr, val)
    return cfg
# --- S3 helpers -----------------------------------------------------------
def s3_client(cfg):
    from botocore.config import Config as BotocoreConfig
    config = BotocoreConfig(retries={"max_attempts": 5, "mode": "adaptive"},
                            connect_timeout=30, read_timeout=60)
    return boto3.client("s3",
                        aws_access_key_id=cfg.aws_access_key_id,
                        aws_secret_access_key=cfg.aws_secret_access_key,
                        region_name=cfg.aws_region,
                        config=config)


def upload_parquet(df, s3, bucket, key, compression="snappy"):
    buf = BytesIO()
    df.to_parquet(buf, index=False, engine="pyarrow", compression=compression,
                  coerce_timestamps="ms", allow_truncated_timestamps=True)
    buf.seek(0)
    s3.put_object(Bucket=bucket, Key=key, Body=buf.getvalue())


def download_parquet(s3, bucket, key):
    obj = s3.get_object(Bucket=bucket, Key=key)
    return pd.read_parquet(BytesIO(obj["Body"].read()))


def list_s3_keys(s3, bucket, prefix):
    keys = []
    paginator = s3.get_paginator("list_objects_v2")
    for page in paginator.paginate(Bucket=bucket, Prefix=prefix):
        for obj in page.get("Contents", []):
            keys.append(obj["Key"])
    return keys


def tickers_from_prefix(s3, bucket, prefix):
    """Ticker symbols from `<prefix>/<TICKER>.parquet` object keys."""
    return [k.split("/")[-1][:-len(".parquet")] for k in list_s3_keys(s3, bucket, prefix)
            if k.endswith(".parquet")]


def duckdb_s3_connect(cfg):
    con = duckdb.connect()
    con.execute("INSTALL httpfs; LOAD httpfs;")
    con.execute(f"SET s3_access_key_id='{cfg.aws_access_key_id}';")
    con.execute(f"SET s3_secret_access_key='{cfg.aws_secret_access_key}';")
    con.execute(f"SET s3_region='{cfg.aws_region}';")
    return con


def spark_session(cfg):
    from pyspark.sql import SparkSession
    spark = (
        SparkSession.builder
        .appName("MarketDataPlatform")
        .config("spark.jars.packages", "org.apache.hadoop:hadoop-aws:3.4.1")
        .config("spark.executor.memory", cfg.spark_executor_memory)
        .config("spark.executor.cores", str(cfg.spark_executor_cores))
        .config("spark.driver.memory", cfg.spark_driver_memory)
        .config("spark.hadoop.fs.s3a.access.key", cfg.aws_access_key_id)
        .config("spark.hadoop.fs.s3a.secret.key", cfg.aws_secret_access_key)
        .config("spark.hadoop.fs.s3a.endpoint", f"s3.{cfg.aws_region}.amazonaws.com")
        .config("spark.local.dir", cfg.spark_tmp)
        .config("spark.hadoop.tmp.dir", cfg.spark_tmp)
        .config("spark.sql.warehouse.dir", f"{cfg.spark_tmp}/warehouse")
        .getOrCreate()
    )
    spark.conf.set("spark.hadoop.fs.s3a.committer.name", "directory")
    spark.conf.set("spark.hadoop.mapreduce.fileoutputcommitter.algorithm.version", "2")
    spark.conf.set("spark.hadoop.fs.s3a.committer.staging.conflict-mode", "append")
    spark.conf.set("spark.sql.debug.maxToStringFields", "100")
    spark.conf.set("spark.sql.autoBroadcastJoinThreshold", "-1")
    spark.conf.set("spark.sql.ansi.enabled", "false")
    spark.conf.set("spark.sql.files.ignoreCorruptFiles", "true")
    spark.conf.set("spark.sql.parquet.mergeSchema", "true")
    spark.sparkContext.setLogLevel("ERROR")
    return spark

# --- Instantiate config + clients -----------------------------
cfg = load_config()
s3 = s3_client(cfg)
CACHE_DIR = Path.cwd() / "data" / "price_cache"
CACHE_DIR.mkdir(parents=True, exist_ok=True)
print("Setup complete")
print(f"Bucket: {cfg.s3_bucket} | Region: {cfg.aws_region}")


In [ ]:
# ============================================================================
# Data source resolution -- prefer the local Kaggle dataset mirror (created
# by 02-02-s3-to-kaggle-dataset: fast, free reads), fall back to S3.
# ============================================================================

import glob as _glob

MIRROR_CANDIDATES = [
    "/kaggle/input/datasets/dsptlp/market-data-s3-dataset/s3_data/parquet_data",
    "/kaggle/input/market-data-s3-dataset/s3_data/parquet_data",
]


def resolve(rel, name="", use_glob=False):
    """Kaggle-mirror path when mounted, else s3a:// URI."""
    short = rel[len("parquet_data/"):] if rel.startswith("parquet_data/") else rel
    for root in MIRROR_CANDIDATES:
        local = os.path.join(root, short, name)
        if use_glob:
            if _glob.glob(local):
                return local
        elif os.path.exists(local):
            return local
    return f"s3://{cfg.s3_bucket}/{rel}/{name}"


In [ ]:
# ============================================================================
# Analysis engine (metrics + robust ranking + reports) -- self-contained helpers (no external package imports)
# ============================================================================


from __future__ import annotations

from pathlib import Path

from scipy.stats import norm

# =============================================================================
# Metrics
# =============================================================================

def _max_drawdown(profit_pct_series: pd.Series) -> float:
    """Worst peak-to-trough decline in cumulative P&L (%)."""
    cum = (1 + profit_pct_series / 100).cumprod()
    peak = cum.cummax()
    dd = (cum - peak) / peak
    return dd.min() * 100


def combine_trades(df_backtest_results: pd.DataFrame) -> pd.DataFrame:
    """Concatenate all per-pair trade DataFrames into one."""
    trades_frames = [r for r in df_backtest_results["trades"].tolist() if r is not None]
    if not trades_frames:
        return pd.DataFrame()
    return pd.concat(trades_frames, ignore_index=True)


def join_market_dod(trades_df: pd.DataFrame, market_dod: pd.DataFrame) -> pd.DataFrame:
    """Join market day-over-day returns onto backtest trades by signal date."""
    trades_df = trades_df.copy()
    trades_df["signal_date"] = pd.to_datetime(trades_df["signal_date"])
    return pd.merge(trades_df, market_dod, left_on="signal_date", right_on="date", how="left")


def compute_pair_metrics(df: pd.DataFrame, min_trades: int = 10) -> pd.DataFrame:
    """
    Per-pair signal quality metrics from backtest trades.

    Parameters
    ----------
    df : DataFrame
        Backtest trades with columns ``leader``, ``follower``, ``correlation``,
        ``leader_gain``, ``profit_pct``, ``signal_date``.
    min_trades : int
        Minimum trades required for a pair to be included.

    Returns
    -------
    DataFrame sorted by ``composite_score`` (IC x win rate) descending.
    """
    df = df.copy()
    df["signal_date"] = pd.to_datetime(df["signal_date"], errors="coerce")
    df = df.sort_values(["leader", "follower", "signal_date"])

    pair_metrics = []
    for (leader, follower, corr), g in df.groupby(["leader", "follower", "correlation"]):
        g = g.dropna(subset=["leader_gain", "profit_pct"])
        n = len(g)
        if n < min_trades:
            continue
        wins = int((g["profit_pct"] > 0).sum())
        gross_profit = g.loc[g["profit_pct"] > 0, "profit_pct"].sum()
        gross_loss = abs(g.loc[g["profit_pct"] <= 0, "profit_pct"].sum())
        ic = g["leader_gain"].corr(g["profit_pct"])
        std = g["profit_pct"].std()
        pair_metrics.append({
            "leader": leader,
            "follower": follower,
            "correlation": corr,
            "n_trades": n,
            "wins": wins,
            "losses": n - wins,
            "win_rate": wins / n,
            "mean_profit_pct": g["profit_pct"].mean(),
            "median_profit_pct": g["profit_pct"].median(),
            "std_profit_pct": std,
            "gross_profit_pct": gross_profit,
            "gross_loss_pct": gross_loss,
            "profit_factor": gross_profit / gross_loss if gross_loss > 0 else np.inf,
            "expected_value_pct": g["profit_pct"].mean(),
            "information_coefficient": ic,
            "max_drawdown_pct": _max_drawdown(g["profit_pct"]),
            "sharpe": (g["profit_pct"].mean() / std * np.sqrt(n)) if std > 0 else 0,
        })

    metrics = pd.DataFrame(pair_metrics)
    if metrics.empty:
        return metrics
    metrics["composite_score"] = metrics["information_coefficient"] * metrics["win_rate"]
    return metrics.sort_values("composite_score", ascending=False)


# =============================================================================
# Robust ranking (Fisher z-transform)
# =============================================================================

def add_robust_ranking(metrics: pd.DataFrame, min_trades_robust: int = 30) -> pd.DataFrame:
    """
    Add a robust ranking using the Fisher z-transform of the IC.

    Adds ``ic_lo_95``, ``ic_hi_95`` (95% CI on the information coefficient)
    and ``robust_score`` = IC * win_rate * sqrt(n_trades).

    Returns the input DataFrame with the new columns merged in.
    """
    robust = metrics[metrics["n_trades"] >= min_trades_robust].copy()
    if robust.empty:
        return metrics

    r = robust["information_coefficient"].clip(-0.999999, 0.999999)
    z = 0.5 * np.log((1 + r) / (1 - r))
    se = 1 / np.sqrt(robust["n_trades"] - 3)
    z_crit = norm.ppf(0.975)

    robust["ic_lo_95"] = (np.exp(2 * (z - z_crit * se)) - 1) / (np.exp(2 * (z - z_crit * se)) + 1)
    robust["ic_hi_95"] = (np.exp(2 * (z + z_crit * se)) - 1) / (np.exp(2 * (z + z_crit * se)) + 1)
    robust["robust_score"] = (
        robust["information_coefficient"] * robust["win_rate"] * np.sqrt(robust["n_trades"])
    )

    merge_cols = ["leader", "follower", "correlation", "ic_lo_95", "ic_hi_95", "robust_score"]
    return metrics.merge(robust[merge_cols], on=["leader", "follower", "correlation"], how="left")


def top_pairs(metrics: pd.DataFrame, metric: str = "robust_score", n: int = 20) -> pd.DataFrame:
    """Return the top ``n`` pairs ranked by ``metric``."""
    return metrics.nlargest(n, metric)


# =============================================================================
# Reporting
# =============================================================================

REPORT_COLUMNS = [
    "leader", "follower", "n_trades", "information_coefficient", "ic_lo_95",
    "win_rate", "expected_value_pct", "profit_factor", "sharpe",
    "max_drawdown_pct", "robust_score",
]


def export_report(
    metrics: pd.DataFrame,
    n: int = 20,
    output_dir: str | Path = ".",
) -> tuple[Path, Path]:
    """
    Export top ``n`` pairs to CSV and a standalone HTML report.

    Returns ``(csv_path, html_path)``.
    """
    top = metrics.nlargest(n, "robust_score")
    cols = [c for c in REPORT_COLUMNS if c in top.columns]

    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    csv_path = output_dir / f"top_{n}_pairs_report.csv"
    top.to_csv(csv_path, index=False)

    html_path = output_dir / f"top_{n}_pairs_report.html"
    html = f"""
    <html><head><title>Top {n} Pairs Report</title>
    <style>
    body {{ font-family: Arial, sans-serif; margin: 40px; }}
    table {{ border-collapse: collapse; width: 100%; }}
    th, td {{ border: 1px solid #ddd; padding: 8px; text-align: right; }}
    th {{ background: #f0f0f0; }}
    td:first-child, td:nth-child(2) {{ text-align: left; }}
    </style></head><body>
    <h1>Top {n} Pairs by Robust Score</h1>
    <p>Generated: {pd.Timestamp.now().strftime('%Y-%m-%d %H:%M')}</p>
    {top[cols].to_html(float_format=lambda x: f'{x:.4f}', index=False)}
    </body></html>
    """
    html_path.write_text(html)
    return csv_path, html_path


# =============================================================================
# Pipeline (notebook 04-02)
# =============================================================================

def load_backtest_trades(cfg: Config, backtest_path: str | None = None) -> pd.DataFrame:
    """Load all backtest trades from S3 (DuckDB httpfs) or a local path."""
    backtest_path = backtest_path or (
        f"s3://{cfg.s3_bucket}/{cfg.backtest_prefix}/*/*"
    )
    con = (
        duckdb_s3_connect(cfg)
        if backtest_path.startswith("s3://")
        else duckdb.connect()
    )
    df = con.execute(
        f"SELECT * FROM read_parquet('{backtest_path}', union_by_name = true)"
    ).df()
    con.close()
    return df








In [ ]:
# ============================================================================
# Load metrics & trades from S3
# ============================================================================

con = duckdb_s3_connect(cfg)

metrics_path = resolve(cfg.backtest_metrics_prefix, "*/*", use_glob=True)
trades_path = resolve(cfg.backtest_prefix, "*/*", use_glob=True)
print("metrics_path:", metrics_path)
print("trades_path:", trades_path)

metrics = con.execute(f"""
    SELECT *
    FROM read_parquet('{metrics_path}', union_by_name = true)
""").df()

trades = con.execute(f"""
    SELECT *
    FROM read_parquet('{trades_path}', union_by_name = true)
""").df()

print(f"Metrics: {len(metrics):,} pairs from {metrics['run_timestamp'].nunique()} runs")
print(f"Trades:  {len(trades):,} total trades")
display(metrics.head())

In [ ]:
# ============================================================================
# Interactive Dashboard: multi-pair comparison
# ============================================================================

metric_dropdown = widgets.Dropdown(
    options=['robust_score', 'information_coefficient', 'ic_lo_95', 'win_rate',
             'expected_value_pct', 'profit_factor', 'sharpe', 'max_drawdown_pct', 'n_trades'],
    value='robust_score',
    description='Rank by:',
    layout=widgets.Layout(width='300px'),
)
topn_slider = widgets.IntSlider(value=20, min=5, max=100, step=5, description='Top N:',
                                layout=widgets.Layout(width='300px'))
mintrades_slider = widgets.IntSlider(value=30, min=10, max=200, step=10,
                                     description='Min trades:', layout=widgets.Layout(width='300px'))
dash_out = widgets.Output()


def update_dashboard(change=None):
    with dash_out:
        clear_output(wait=True)
        metric = metric_dropdown.value
        top_n = topn_slider.value
        min_t = mintrades_slider.value
        df = metrics[metrics['n_trades'] >= min_t].nlargest(top_n, metric)

        fig = px.bar(
            df,
            x=df['leader'] + '→' + df['follower'],
            y=metric,
            hover_data=['n_trades', 'win_rate', 'information_coefficient',
                        'expected_value_pct', 'profit_factor', 'sharpe'],
            title=f'Top {top_n} pairs by {metric} (min trades={min_t})',
            labels={'x': 'Pair', 'y': metric},
            color=metric,
            color_continuous_scale='RdYlGn',
        )
        fig.update_layout(xaxis_tickangle=-45, height=500)
        fig.show()

        display(df[['leader', 'follower', 'n_trades', 'information_coefficient',
                    'ic_lo_95', 'win_rate', 'expected_value_pct', 'profit_factor',
                    'sharpe', 'max_drawdown_pct', 'robust_score']])


for w in (metric_dropdown, topn_slider, mintrades_slider):
    w.observe(update_dashboard, names='value')

ui = widgets.VBox([widgets.HBox([metric_dropdown, topn_slider, mintrades_slider]), dash_out])
display(ui)
update_dashboard()

In [ ]:
# ============================================================================
# Pair Drilldown: trade-by-trade analysis
# ============================================================================

pair_options = [
    (f"{r['leader']} → {r['follower']} (n={int(r['n_trades'])}, IC={r['information_coefficient']:.3f})",
     (r['leader'], r['follower']))
    for _, r in metrics.nlargest(100, 'robust_score').iterrows()
]
pair_dropdown = widgets.Dropdown(options=pair_options, description='Pair:',
                                 layout=widgets.Layout(width='500px'))
drilldown_out = widgets.Output()


def load_daily_prices(leader, follower):
    cache_file = CACHE_DIR / f"{leader}_{follower}_daily.parquet"
    if cache_file.exists():
        return pd.read_parquet(cache_file)
    prices_path = resolve(cfg.minute_summary_prefix, "data.parquet")
    pdf = con.execute(f"""
        SELECT symbol, trade_date, close
        FROM read_parquet('{prices_path}')
        WHERE symbol IN ('{leader}', '{follower}') AND rn_desc = 1
        ORDER BY symbol, trade_date
    """).df()
    pdf['trade_date'] = pd.to_datetime(pdf['trade_date'])
    pivot = pdf.pivot(index='trade_date', columns='symbol', values='close').dropna()
    pivot.to_parquet(cache_file)
    return pivot


def show_pair_analysis(leader, follower):
    with drilldown_out:
        clear_output(wait=True)
        pt = (trades[(trades['leader'] == leader) & (trades['follower'] == follower)]
              .sort_values('signal_date').copy())
        if len(pt) == 0:
            print(f"No trades found for {leader} → {follower}")
            return

        print(f"{'=' * 60}")
        print(f"{leader} → {follower}  |  {len(pt)} trades")
        print(f"{'=' * 60}")

        m = metrics[(metrics['leader'] == leader) & (metrics['follower'] == follower)].iloc[0]
        print(f"IC: {m['information_coefficient']:.4f}  |  95% CI: [{m['ic_lo_95']:.4f}, {m.get('ic_hi_95', 0):.4f}]")
        print(f"Win Rate: {m['win_rate']:.1%}  |  EV: {m['expected_value_pct']:.2f}%  |  PF: {m['profit_factor']:.2f}")
        print(f"Sharpe: {m['sharpe']:.2f}  |  MaxDD: {m['max_drawdown_pct']:.1f}%  |  Robust: {m['robust_score']:.2f}")

        fig = make_subplots(
            rows=2, cols=2,
            subplot_titles=('Signal vs Outcome', 'Cumulative P&L',
                            'P&L Distribution', 'Rolling IC (30-trade)'),
        )
        fig.add_trace(go.Scatter(x=pt['leader_gain'], y=pt['profit_pct'], mode='markers',
                                 marker=dict(color='steelblue', size=6, opacity=0.6),
                                 name='Trades',
                                 hovertemplate='Leader: %{x:.2f}%<br>Profit: %{y:.2f}%'),
                      row=1, col=1)
        z = np.polyfit(pt['leader_gain'], pt['profit_pct'], 1)
        x_line = np.linspace(pt['leader_gain'].min(), pt['leader_gain'].max(), 100)
        fig.add_trace(go.Scatter(x=x_line, y=z[0] * x_line + z[1], mode='lines',
                                 line=dict(color='red', dash='dash', width=2),
                                 name=f'β={z[0]:.3f}'), row=1, col=1)

        cum = (1 + pt['profit_pct'] / 100).cumprod()
        fig.add_trace(go.Scatter(y=cum, mode='lines', line=dict(color='steelblue'),
                                 name='Cum P&L'), row=1, col=2)
        fig.add_hline(y=1, line_dash='dash', line_color='black', row=1, col=2)

        fig.add_trace(go.Histogram(x=pt['profit_pct'], nbinsx=30, marker_color='steelblue',
                                   opacity=0.7, name='P&L'), row=2, col=1)
        fig.add_vline(x=0, line_dash='dash', line_color='black', row=2, col=1)

        window = min(30, len(pt) // 2)
        if window >= 10:
            rolling_ic = pt['leader_gain'].rolling(window).corr(pt['profit_pct'])
            fig.add_trace(go.Scatter(y=rolling_ic, mode='lines', line=dict(color='coral'),
                                     name=f'Rolling IC ({window})'), row=2, col=2)
            fig.add_hline(y=0, line_dash='dash', line_color='black', row=2, col=2)

        fig.update_layout(height=800, showlegend=True,
                          title_text=f'{leader} → {follower} Deep Dive')
        fig.show()

        display(pt[['signal_date', 'leader_gain', 'buy_date', 'sell_date',
                    'profit_pct', 'correlation']].head(30))

        print("\nLoading daily prices...")
        prices = load_daily_prices(leader, follower)
        norm = prices / prices.iloc[0] * 100

        fig2 = make_subplots(rows=2, cols=1, shared_xaxes=True,
                             subplot_titles=('Raw Prices', 'Normalized (start=100)'))
        for col, color in [(leader, 'steelblue'), (follower, 'coral')]:
            fig2.add_trace(go.Scatter(x=prices.index, y=prices[col], mode='lines',
                                      line=dict(color=color, width=1), name=f'{col}'),
                           row=1, col=1)
            fig2.add_trace(go.Scatter(x=norm.index, y=norm[col], mode='lines',
                                      line=dict(color=color, width=1), name=f'{col}',
                                      showlegend=False), row=2, col=1)
        fig2.add_hline(y=100, line_dash='dash', line_color='gray', row=2, col=1)
        fig2.update_layout(height=600, hovermode='x unified')
        fig2.show()

        lr = prices[leader].pct_change().dropna()
        fr = prices[follower].pct_change().dropna()
        aligned = pd.concat([lr, fr], axis=1, keys=['leader', 'follower']).dropna()
        print(f"Daily return correlation: {aligned['leader'].corr(aligned['follower']):.4f}")


def on_pair_change(change):
    val = change['new']
    leader, follower = val[1] if isinstance(val, tuple) and len(val) == 2 else val
    show_pair_analysis(leader, follower)


pair_dropdown.observe(on_pair_change, names='value')
display(widgets.VBox([pair_dropdown, drilldown_out]))
leader, follower = pair_options[0][1]
show_pair_analysis(leader, follower)

In [ ]:
# ============================================================================
# Threshold Sensitivity Heatmap (Top N pairs)
# ============================================================================

top_n_heatmap = widgets.IntSlider(value=10, min=5, max=30, step=5,
                                  description='Top N pairs:', layout=widgets.Layout(width='300px'))
heatmap_out = widgets.Output()


def plot_threshold_heatmap(top_n=10):
    with heatmap_out:
        clear_output(wait=True)
        top_pairs = metrics.nlargest(top_n, 'robust_score')
        thresholds = np.arange(1, 21, 1)

        heatmap_data = []
        for _, row in top_pairs.iterrows():
            ldr, fll = row['leader'], row['follower']
            pt = trades[(trades['leader'] == ldr) & (trades['follower'] == fll)]
            row_data = []
            for t in thresholds:
                sub = pt[pt['leader_gain'] >= t]
                row_data.append((sub['profit_pct'] > 0).mean() * 100 if len(sub) >= 5 else np.nan)
            heatmap_data.append(row_data)

        heatmap_df = pd.DataFrame(
            heatmap_data,
            index=[f"{r['leader']}→{r['follower']}" for _, r in top_pairs.iterrows()],
            columns=[f"≥{t}%" for t in thresholds],
        )
        fig = px.imshow(
            heatmap_df.values, x=heatmap_df.columns, y=heatmap_df.index,
            color_continuous_scale='RdYlGn', aspect='auto',
            labels=dict(x='Leader Gain Threshold', y='Pair', color='Win Rate %'),
            title=f'Win Rate % by Leader Gain Threshold (Top {top_n} pairs)',
        )
        fig.update_layout(height=400 + top_n * 25)
        fig.show()


top_n_heatmap.observe(lambda c: plot_threshold_heatmap(c['new']), names='value')
display(widgets.VBox([top_n_heatmap, heatmap_out]))
plot_threshold_heatmap(10)

In [ ]:
# ============================================================================
# Rolling IC Stability Analysis
# ============================================================================

stab_pair_dropdown = widgets.Dropdown(options=pair_options, description='Pair:',
                                      layout=widgets.Layout(width='500px'))
window_slider = widgets.IntSlider(value=30, min=10, max=100, step=10, description='Window:',
                                  layout=widgets.Layout(width='300px'))
stab_out = widgets.Output()


def plot_rolling_ic(leader, follower, window=30):
    with stab_out:
        clear_output(wait=True)
        pt = (trades[(trades['leader'] == leader) & (trades['follower'] == follower)]
              .sort_values('signal_date'))
        if len(pt) < window * 2:
            print(f"Not enough trades ({len(pt)}) for window {window}")
            return

        rolling_ic = pt['leader_gain'].rolling(window).corr(pt['profit_pct'])
        rolling_wr = pt['profit_pct'].rolling(window).apply(lambda x: (x > 0).mean())
        rolling_ev = pt['profit_pct'].rolling(window).mean()

        fig = make_subplots(rows=3, cols=1, shared_xaxes=True,
                            subplot_titles=('Rolling IC', 'Rolling Win Rate', 'Rolling EV %'))
        fig.add_trace(go.Scatter(y=rolling_ic, mode='lines', name='IC',
                                 line=dict(color='steelblue')), row=1, col=1)
        fig.add_hline(y=0, line_dash='dash', row=1, col=1)
        fig.add_hline(y=pt['leader_gain'].corr(pt['profit_pct']), line_dash='dot',
                      line_color='red', row=1, col=1, annotation_text='Full-sample IC')
        fig.add_trace(go.Scatter(y=rolling_wr * 100, mode='lines', name='Win Rate',
                                 line=dict(color='coral')), row=2, col=1)
        fig.add_hline(y=50, line_dash='dash', row=2, col=1)
        fig.add_trace(go.Scatter(y=rolling_ev, mode='lines', name='EV',
                                 line=dict(color='green')), row=3, col=1)
        fig.add_hline(y=0, line_dash='dash', row=3, col=1)
        fig.update_layout(height=700, hovermode='x unified',
                          title=f'{leader} → {follower} Rolling Metrics (window={window})')
        fig.show()

        ic_vals = rolling_ic.dropna()
        print(f"IC stability: mean={ic_vals.mean():.4f}, std={ic_vals.std():.4f}, "
              f"min={ic_vals.min():.4f}, max={ic_vals.max():.4f}")
        print(f"% windows with positive IC: {(ic_vals > 0).mean():.1%}")


stab_pair_dropdown.observe(
    lambda c: plot_rolling_ic(c['new'][1][0], c['new'][1][1], window_slider.value),
    names='value')
window_slider.observe(lambda c: plot_rolling_ic(
    stab_pair_dropdown.value[1][0], stab_pair_dropdown.value[1][1], c['new']), names='value')

display(widgets.VBox([widgets.HBox([stab_pair_dropdown, window_slider]), stab_out]))
leader, follower = pair_options[0][1]
plot_rolling_ic(leader, follower, 30)

In [ ]:
# ============================================================================
# Export Report: HTML + CSV for top pairs
# ============================================================================

export_n = widgets.IntSlider(value=20, min=5, max=100, step=5,
                             description='Top N to export:', layout=widgets.Layout(width='400px'))
export_btn = widgets.Button(description='Generate Report', button_style='success',
                            layout=widgets.Layout(width='200px'))
export_out = widgets.Output()


def generate_report(b):
    with export_out:
        clear_output(wait=True)
        n = export_n.value
        csv_path, html_path = export_report(metrics, n=n, output_dir="reports")
        print(f"CSV saved:  {csv_path}")
        print(f"HTML saved: {html_path}")
        display(metrics.nlargest(n, 'robust_score')[
            ['leader', 'follower', 'n_trades', 'information_coefficient', 'ic_lo_95',
             'win_rate', 'expected_value_pct', 'profit_factor', 'sharpe',
             'max_drawdown_pct', 'robust_score']])


export_btn.on_click(generate_report)
display(widgets.VBox([widgets.HBox([export_n, export_btn]), export_out]))

In [ ]:
# ============================================================================
# HEADLESS TEST -- run core logic without widgets (for CI / nbconvert)
# ============================================================================

print("=== Headless Test ===")

# Test 1: Dashboard logic
df = metrics[metrics['n_trades'] >= 30].nlargest(20, 'robust_score')
print(f"Top 20 by robust_score (min 30 trades): {len(df)} rows")
print(df[['leader', 'follower', 'robust_score', 'information_coefficient', 'win_rate']].head())

# Test 2: Recompute metrics from trades via the analysis library
recomputed = compute_pair_metrics(trades, min_trades=10)
recomputed = add_robust_ranking(recomputed, min_trades_robust=30)
print(f"\nRecomputed metrics from trades: {len(recomputed):,} pairs")
print(recomputed.nlargest(5, 'robust_score')[['leader', 'follower', 'robust_score']])

# Test 3: Export logic
top20 = metrics.nlargest(20, 'robust_score')
print(f"\nExport test: {len(top20)} rows ready for CSV/HTML")

print("\n=== All headless tests passed ===")